In [9]:

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TORCH_USE_CUDA_DSA']   = '1'

import copy
import itertools
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch_geometric.nn import GCNConv, MessagePassing
from torch_geometric.utils import add_self_loops, softmax

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score
from sklearn.metrics.pairwise import cosine_similarity

from tqdm import trange

print('Imports OK')
print(f'PyTorch   : {torch.__version__}')
print(f'CUDA avail: {torch.cuda.is_available()}')

csv_path      = 'LAML_data_complete_modalities_preprocessed.csv'
omics_prefixes = ('gex_', 'cnv_', 'meth_', 'mirna_hsa-', 'mut_', 'rppa_')
label_col      = 'OS'

cols       = pd.read_csv(csv_path, nrows=0).columns.tolist()
omics_cols = [c for c in cols if c.startswith(omics_prefixes)]

MAX_GENES = 6000
if len(omics_cols) > MAX_GENES:
    np.random.seed(42)
    omics_cols = np.random.choice(omics_cols, MAX_GENES, replace=False).tolist()

df      = pd.read_csv(csv_path, usecols=omics_cols + [label_col])
y_raw   = df[label_col].values.astype(np.float32)
X_np    = df[omics_cols].values.astype(np.float32)

var  = X_np.var(axis=0)
X_np = X_np[:, var > 0]
X_np = np.nan_to_num(X_np)
X_np = StandardScaler().fit_transform(X_np)

K           = 4
gene_splits = np.array_split(np.arange(X_np.shape[1]), K)
X_list      = [torch.tensor(X_np[:, idx], dtype=torch.float32) for idx in gene_splits]

y = torch.tensor(y_raw, dtype=torch.float32).view(-1, 1)

print('Preprocessing complete.')
print(f'  Patients : {X_np.shape[0]}')
print(f'  Features : {X_np.shape[1]}')
print(f'  Omics splits: {[x.shape[1] for x in X_list]}')
print(f'  Label distribution:\n{pd.Series(y_raw).value_counts()}')


def build_fuzzy_graph(X, k=10):
    if isinstance(X, torch.Tensor):
        X = X.numpy()
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, 0)
    edges, weights = [], []
    for i in range(sim.shape[0]):
        topk = np.argsort(sim[i])[-k:]
        for j in topk:
            edges.append([i, int(j)])
            weights.append(float(sim[i, j]))
            edges.append([int(j), i])
            weights.append(float(sim[i, j]))
    edge_index  = torch.tensor(edges,   dtype=torch.long).t().contiguous()
    edge_weight = torch.tensor(weights, dtype=torch.float32)
    return edge_index, edge_weight


edge_index, edge_weight = build_fuzzy_graph(X_np, k=10)

print(f'Graph built.')
print(f'  Nodes : {X_np.shape[0]}')
print(f'  Edges : {edge_index.shape[1]}')


class FuzzyCoverConv(MessagePassing):
    def __init__(self, in_channels, out_channels, heads=1, dropout=0.0):
        super().__init__(aggr='add', node_dim=0)
        self.heads        = heads
        self.out_channels = out_channels
        self.dropout      = dropout
        self.lin     = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.att_src = nn.Parameter(torch.Tensor(1, heads, out_channels))
        self.att_dst = nn.Parameter(torch.Tensor(1, heads, out_channels))
        self.bias    = nn.Parameter(torch.Tensor(out_channels))
        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)
        nn.init.zeros_(self.bias)

    def forward(self, x, edge_index, edge_weight=None):
        x   = self.lin(x).view(-1, self.heads, self.out_channels)
        out = self.propagate(edge_index, x=x, edge_weight=edge_weight)
        return out.mean(dim=1) + self.bias

    def message(self, x_i, x_j, edge_weight, index, ptr, size_i):
        alpha = (x_i * self.att_src).sum(-1) + (x_j * self.att_dst).sum(-1)
        alpha = F.leaky_relu(alpha, 0.2)
        if edge_weight is not None:
            alpha = alpha * edge_weight.view(-1, 1)
        alpha = softmax(alpha, index, ptr, size_i)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)
        return x_j * alpha.unsqueeze(-1)


class FGC_GNN(nn.Module):
    def __init__(self, in_dims, hidden_channels=128, out_channels=1,
                 heads=4, dropout=0.3):
        super().__init__()
        self.dropout_p  = dropout
        self.num_omics  = len(in_dims)

        self.omics_gcns = nn.ModuleList([
            FuzzyCoverConv(d, hidden_channels, heads=heads, dropout=dropout)
            for d in in_dims
        ])
        self.res_proj = nn.ModuleList([
            nn.Linear(d, hidden_channels) for d in in_dims
        ])
        self.gates = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_channels, hidden_channels // 4),
                nn.ReLU(),
                nn.Linear(hidden_channels // 4, 1),
                nn.Sigmoid(),
            ) for _ in range(self.num_omics)
        ])
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden_channels) for _ in range(self.num_omics)
        ])
        self.cross_attn = nn.MultiheadAttention(
            hidden_channels, num_heads=heads, dropout=dropout, batch_first=True
        )

        fused = hidden_channels * self.num_omics
        self.fc1 = nn.Linear(fused, hidden_channels * 2)
        self.bn1 = nn.BatchNorm1d(hidden_channels * 2)
        self.fc2 = nn.Linear(hidden_channels * 2, hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        self.fc3 = nn.Linear(hidden_channels, out_channels)

    def forward(self, X_list, edge_index, edge_weight=None):
        omics_embs = []
        for gcn, res, gate, ln, X in zip(
            self.omics_gcns, self.res_proj, self.gates, self.layer_norms, X_list
        ):
            h = gcn(X, edge_index, edge_weight=edge_weight) + res(X)
            h = F.relu(ln(h))
            h = F.dropout(h, p=self.dropout_p, training=self.training)
            h = gate(h) * h
            omics_embs.append(h)

        stack = torch.stack(omics_embs, dim=1)
        attn_out, _ = self.cross_attn(stack, stack, stack)
        stack = stack + attn_out

        h = stack.reshape(stack.size(0), -1)
        h = F.dropout(F.relu(self.bn1(self.fc1(h))), p=self.dropout_p, training=self.training)
        h = F.dropout(F.relu(self.bn2(self.fc2(h))), p=self.dropout_p, training=self.training)
        return torch.sigmoid(self.fc3(h))


print('FGC_GNN defined.')
total_params = sum(p.numel() for p in FGC_GNN([x.shape[1] for x in X_list]).parameters())
print(f'Param count (default config): {total_params:,}')


class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, preds, targets):
        preds   = preds.float().view(-1)
        targets = targets.float().view(-1)
        bce  = F.binary_cross_entropy(preds, targets, reduction='none')
        pt   = torch.exp(-bce)
        loss = self.alpha * (1 - pt) ** self.gamma * bce
        return loss.mean()


print('FocalLoss defined.')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

X_list_gpu   = [X.float().to(device) for X in X_list]
edge_idx_gpu = edge_index.long().to(device)
edge_wt_gpu  = edge_weight.float().to(device)
y_gpu        = y.float().to(device)

in_dims = [x.shape[1] for x in X_list_gpu]

model = FGC_GNN(in_dims, hidden_channels=64, out_channels=1).to(device)

optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=5e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=400)
criterion = FocalLoss()

losses, accs, f1s, aucs = [], [], [], []
best_loss = float('inf')

for epoch in trange(400, desc='Sanity-check training'):
    model.train()
    optimizer.zero_grad()

    probs = model(X_list_gpu, edge_idx_gpu, edge_wt_gpu)
    loss  = criterion(probs, y_gpu)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    with torch.no_grad():
        p_np   = probs.cpu().numpy().ravel()
        y_np   = y_gpu.cpu().numpy().ravel()
        
        # ── Tune threshold on training data ──
        thresholds = np.linspace(0.1, 0.9, 50)
        f1_scores = [f1_score(y_np, (p_np >= t).astype(int), zero_division=0) for t in thresholds]
        best_thr = thresholds[np.argmax(f1_scores)]
        
        y_pred = (p_np >= best_thr).astype(int)
        acc    = (y_pred == y_np).mean()
        f1     = f1_score(y_np, y_pred, zero_division=0)
        auc    = roc_auc_score(y_np, p_np)

    losses.append(loss.item())
    accs.append(acc)
    f1s.append(f1)
    aucs.append(auc)

    if loss.item() < best_loss:
        best_loss = loss.item()
        torch.save(model.state_dict(), 'FGC_LAML_GNN_best.pt')

pd.DataFrame({
    'Epoch': range(1, 401),
    'Loss': losses, 'Accuracy': accs, 'F1': f1s, 'AUC': aucs,
}).to_csv('FGC_GNN_LAML_sanity_metrics.csv', index=False)

print(f'\nBest loss : {best_loss:.4f}')
print(f'Final AUC : {aucs[-1]:.4f}')


#  PARAM GRID 
PARAM_GRID = {
    'hidden_channels': [32, 64, 128],
    'lr'             : [1e-3, 2e-4],
    'weight_decay'   : [1e-3, 1e-4],
    'dropout'        : [0.3, 0.5],
    'focal_gamma'    : [0.5, 1],
    'focal_alpha'    : [0.5, 0.75],
    'epochs'         : [100, 200, 400],
}

keys   = list(PARAM_GRID.keys())
combos = [dict(zip(keys, v)) for v in itertools.product(*PARAM_GRID.values())]
print(f'Total combinations: {len(combos)}')


def find_best_threshold(y_true, y_prob):
    """Find optimal threshold that maximizes F1 score."""
    thresholds = np.linspace(0.1, 0.9, 81)
    f1_scores = []
    for thr in thresholds:
        pred = (y_prob >= thr).astype(int)
        f1_scores.append(f1_score(y_true, pred, zero_division=0))
    best_idx = np.argmax(f1_scores)
    return thresholds[best_idx], f1_scores[best_idx]


def evaluate_subset(model, X_list_gpu, edge_index, edge_weight, y_gpu, idx_t, threshold=0.5):
    model.eval()
    with torch.no_grad():
        out  = model(X_list_gpu, edge_index, edge_weight)
        y_np = y_gpu[idx_t].cpu().numpy().ravel()
        p_np = out[idx_t].cpu().numpy().ravel()
        pred = (p_np >= threshold).astype(int)
    return {
        'auc': roc_auc_score(y_np, p_np),
        'f1' : f1_score(y_np, pred, zero_division=0),
        'acc': accuracy_score(y_np, pred),
        'precision': precision_score(y_np, pred, zero_division=0),
    }


def run_repeated_cv(hp, X_list_gpu, edge_index, edge_weight, y_gpu, in_dims,
                    n_splits=5, n_repeats=3, seed=42, patience=20):
    """Run repeated stratified k-fold cross-validation with threshold tuning."""
    y_np = y_gpu.cpu().numpy().ravel()
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=seed)
    
    fold_aucs, fold_f1s, fold_accs, fold_precs, fold_thrs = [], [], [], [], []

    for fold_idx, (train_idx, val_idx) in enumerate(rskf.split(np.zeros(len(y_np)), y_np)):
        tr_t = torch.tensor(train_idx, dtype=torch.long, device=device)
        vl_t = torch.tensor(val_idx,   dtype=torch.long, device=device)

        model = FGC_GNN(
            in_dims,
            hidden_channels = int(hp['hidden_channels']),
            out_channels    = 1,
            dropout         = hp['dropout'],
        ).to(device)

        optimizer  = optim.AdamW(model.parameters(),
                                 lr=hp['lr'], weight_decay=hp['weight_decay'])
        scheduler  = optim.lr_scheduler.CosineAnnealingLR(
                         optimizer, T_max=int(hp['epochs']))
        criterion  = FocalLoss(alpha=hp['focal_alpha'], gamma=hp['focal_gamma'])
        amp_scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

        y_tr = y_gpu[tr_t]

        # Early stopping state 
        best_val_f1      = -1.0
        best_fold_state  = None
        best_fold_thr    = 0.5
        epochs_no_improve = 0

        for epoch in range(int(hp['epochs'])):
            model.train()
            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                out = model(X_list_gpu, edge_index, edge_weight)

            loss = criterion(out[tr_t].float(), y_tr.float())
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            amp_scaler.step(optimizer)
            amp_scaler.update()
            scheduler.step()

            
            model.eval()
            with torch.no_grad():
                val_probs = out[vl_t].cpu().numpy().ravel()
                val_true  = y_gpu[vl_t].cpu().numpy().ravel()
            
            val_thr, val_f1 = find_best_threshold(val_true, val_probs)

            if val_f1 > best_val_f1:
                best_val_f1       = val_f1
                best_fold_state   = copy.deepcopy(model.state_dict())
                best_fold_thr     = val_thr
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= patience:
                break

        # Restore best checkpoint for this fold 
        model.load_state_dict(best_fold_state)
        m = evaluate_subset(model, X_list_gpu, edge_index, edge_weight,
                            y_gpu, vl_t, threshold=best_fold_thr)
        fold_aucs.append(m['auc'])
        fold_f1s.append(m['f1'])
        fold_accs.append(m['acc'])
        fold_precs.append(m['precision'])
        fold_thrs.append(best_fold_thr)

    return {
        'mean_auc': float(np.mean(fold_aucs)),
        'std_auc' : float(np.std(fold_aucs)),
        'mean_f1' : float(np.mean(fold_f1s)),
        'std_f1'  : float(np.std(fold_f1s)),
        'mean_acc': float(np.mean(fold_accs)),
        'std_acc' : float(np.std(fold_accs)),
        'mean_precision': float(np.mean(fold_precs)),
        'std_precision' : float(np.std(fold_precs)),
        'mean_threshold': float(np.mean(fold_thrs)),
        'std_threshold' : float(np.std(fold_thrs)),
    }


print('CV helpers defined.')

DATASET = 'LAML'

y_gpu = y.float().to(device)
assert set(y_gpu.unique().tolist()).issubset({0.0, 1.0}), \
    f"Unexpected labels: {y_gpu.unique()}"

X_list_gpu   = [X.float().to(device) for X in X_list]
edge_idx_gpu = edge_index.long().to(device)
edge_wt_gpu  = edge_weight.float().to(device)

assert edge_idx_gpu.max() < X_list_gpu[0].shape[0], "Edge index out of bounds!"

in_dims = [X.shape[1] for X in X_list_gpu]
print(f'Dataset  : {DATASET}')
print(f'Device   : {device}')
print(f'Nodes    : {X_list_gpu[0].shape[0]}')
print(f'Edges    : {edge_idx_gpu.shape[1]}')
print(f'in_dims  : {in_dims}')
print(f'Combos   : {len(combos)}\n')

cv_results = []

# Use Repeated CV (5-fold × 3 repeats = 15 folds)
N_SPLITS  = 5
N_REPEATS = 3

print(f'Using Repeated Stratified CV: {N_SPLITS}-fold × {N_REPEATS} repeats = {N_SPLITS * N_REPEATS} total folds\n')

for run_idx, hp in enumerate(combos):
    result = run_repeated_cv(
        hp          = hp,
        X_list_gpu  = X_list_gpu,
        edge_index  = edge_idx_gpu,
        edge_weight = edge_wt_gpu,
        y_gpu       = y_gpu,
        in_dims     = in_dims,
        n_splits    = N_SPLITS,
        n_repeats   = N_REPEATS,
        seed        = 42,
        patience    = 10,
    )
    cv_results.append({**hp, **result})
    print(
        f'[{run_idx+1:>4}/{len(combos)}] '
        f'hidden={hp["hidden_channels"]:>3} '
        f'lr={hp["lr"]:.0e} '
        f'wd={hp["weight_decay"]:.0e} '
        f'drop={hp["dropout"]} '
        f'epochs={hp["epochs"]:>4} '
        f'F1={result["mean_f1"]:.4f}±{result["std_f1"]:.4f} '
        f'ACC={result["mean_acc"]:.4f}±{result["std_acc"]:.4f} '
        f'THR={result["mean_threshold"]:.3f}±{result["std_threshold"]:.3f}'
    )

cv_df = (
    pd.DataFrame(cv_results)
    .sort_values('mean_f1', ascending=False)
    .reset_index(drop=True)
)
cv_df.to_csv(f'FGC_GNN_{DATASET}_cv_results.csv', index=False)

display_cols = [
    'hidden_channels', 'lr', 'weight_decay', 'dropout',
    'focal_gamma', 'focal_alpha', 'epochs',
    'mean_auc', 'std_auc', 'mean_f1', 'std_f1', 
    'mean_acc', 'std_acc', 'mean_precision', 'std_precision',
    'mean_threshold', 'std_threshold',
]
print(f'\n--- Top 5 configurations ({DATASET}) ---')
print(cv_df[display_cols].head(5).to_string(index=True))


best_hp = {k: cv_df.loc[0, k] for k in PARAM_GRID.keys()}
best_hp['hidden_channels'] = int(best_hp['hidden_channels'])
best_hp['epochs']          = int(best_hp['epochs'])
best_hp['focal_gamma']     = float(best_hp['focal_gamma'])
best_hp['focal_alpha']     = float(best_hp['focal_alpha'])

print('Best hyperparameters (based on F1):')
for k, v in best_hp.items():
    print(f'  {k:20s}: {v}')
print(f'\nExpected CV F1  : {cv_df.loc[0,"mean_f1"]:.4f} ± {cv_df.loc[0,"std_f1"]:.4f}')
print(f'Expected CV ACC : {cv_df.loc[0,"mean_acc"]:.4f} ± {cv_df.loc[0,"std_acc"]:.4f}')
print(f'Expected CV AUC : {cv_df.loc[0,"mean_auc"]:.4f} ± {cv_df.loc[0,"std_auc"]:.4f}')
print(f'Expected CV THR : {cv_df.loc[0,"mean_threshold"]:.3f} ± {cv_df.loc[0,"std_threshold"]:.3f}')

y_np_flat = y_gpu.cpu().numpy().ravel()
all_idx   = np.arange(len(y_np_flat))

# Run final training and testing 10 times with  CV 
NUM_RUNS = 10
FINAL_PATIENCE = 10

all_run_results = []

for run_num in range(NUM_RUNS):
    print(f'\n{"="*70}')
    print(f'  RUN {run_num + 1}/{NUM_RUNS}')
    print(f'{"="*70}')
    
    # Split 
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42 + run_num)
    train_idx, test_idx = next(skf.split(all_idx, y_np_flat))
    
    train_t = torch.tensor(train_idx, dtype=torch.long, device=device)
    test_t  = torch.tensor(test_idx,  dtype=torch.long, device=device)

    print(f'Train samples : {len(train_idx)}')
    print(f'Test  samples : {len(test_idx)}')
    print(f'Train label dist: {np.bincount(y_np_flat[train_idx].astype(int))}')
    print(f'Test  label dist: {np.bincount(y_np_flat[test_idx].astype(int))}')

    final_model = FGC_GNN(
        in_dims         = in_dims,
        hidden_channels = best_hp['hidden_channels'],
        out_channels    = 1,
        dropout         = best_hp['dropout'],
    ).to(device)

    final_optimizer = optim.AdamW(
        final_model.parameters(),
        lr=best_hp['lr'], weight_decay=best_hp['weight_decay'],
    )
    final_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        final_optimizer, T_max=best_hp['epochs']
    )
    final_criterion = FocalLoss(alpha=best_hp['focal_alpha'],
                                gamma=best_hp['focal_gamma'])

    y_train_final    = y_gpu[train_t]
    train_losses, train_aucs, train_f1s, train_accs, train_thrs = [], [], [], [], []
    best_train_f1    = 0.0
    best_model_state = None
    best_train_thr   = 0.5
    final_no_improve = 0

    print(f'Training final model | {best_hp["epochs"]} epochs | patience={FINAL_PATIENCE}')

    for epoch in trange(best_hp['epochs'], desc=f'Final Training (Run {run_num + 1})'):
        final_model.train()
        final_optimizer.zero_grad()

        out  = final_model(X_list_gpu, edge_idx_gpu, edge_wt_gpu)
        loss = final_criterion(out[train_t].float(), y_train_final.float())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_model.parameters(), 2.0)
        final_optimizer.step()
        final_scheduler.step()

        final_model.eval()
        with torch.no_grad():
            p_tr   = out[train_t].cpu().numpy().ravel()
            y_tr   = y_train_final.cpu().numpy().ravel()
            
            # Find best threshold on training set 
            thr_tr, f1_tr = find_best_threshold(y_tr, p_tr)
            pred_tr = (p_tr >= thr_tr).astype(int)
            
            auc_tr = roc_auc_score(y_tr, p_tr)
            acc_tr = accuracy_score(y_tr, pred_tr)

        train_losses.append(loss.item())
        train_aucs.append(auc_tr)
        train_f1s.append(f1_tr)
        train_accs.append(acc_tr)
        train_thrs.append(thr_tr)

        if f1_tr > best_train_f1:
            best_train_f1    = f1_tr
            best_model_state = copy.deepcopy(final_model.state_dict())
            best_train_thr   = thr_tr
            final_no_improve = 0
        else:
            final_no_improve += 1

        if final_no_improve >= FINAL_PATIENCE:
            print(f'\nEarly stopping triggered at epoch {epoch + 1}')
            break

    final_model.load_state_dict(best_model_state)
    torch.save(best_model_state, f'FGC_GNN_{DATASET}_best_final_run{run_num + 1}.pt')

    pd.DataFrame({
        'Epoch'    : range(1, len(train_losses) + 1),
        'Loss'     : train_losses,
        'Train_AUC': train_aucs,
        'Train_F1' : train_f1s,
        'Train_ACC': train_accs,
        'Train_THR': train_thrs,
    }).to_csv(f'FGC_GNN_{DATASET}_final_training_curve_run{run_num + 1}.csv', index=False)

    print(f'\nBest training F1  : {best_train_f1:.4f}')
    print(f'Best training AUC : {max(train_aucs):.4f}')
    print(f'Best training ACC : {max(train_accs):.4f}')
    print(f'Best threshold    : {best_train_thr:.3f}')

    # Test evaluation with tuned threshold 
    final_model.eval()
    with torch.no_grad():
        full_out  = final_model(X_list_gpu, edge_idx_gpu, edge_wt_gpu)
        y_test_np = y_gpu[test_t].cpu().numpy().ravel()
        p_test_np = full_out[test_t].cpu().numpy().ravel()
        
        #  Use threshold from training 
        y_pred_np = (p_test_np >= best_train_thr).astype(int)

    test_auc  = roc_auc_score(y_test_np, p_test_np)
    test_f1   = f1_score(y_test_np, y_pred_np, zero_division=0)
    test_acc  = accuracy_score(y_test_np, y_pred_np)
    test_prec = precision_score(y_test_np, y_pred_np, zero_division=0)

    all_run_results.append({
        'run': run_num + 1,
        'test_auc': test_auc,
        'test_f1': test_f1,
        'test_acc': test_acc,
        'test_precision': test_prec,
        'threshold': best_train_thr,
    })

    print(f'\nRun {run_num + 1} Test Results (threshold={best_train_thr:.3f}):')
    print(f'  AUC       : {test_auc:.4f}')
    print(f'  F1 Score  : {test_f1:.4f}')
    print(f'  Accuracy  : {test_acc:.4f}')
    print(f'  Precision : {test_prec:.4f}')

    pd.DataFrame({
        'sample_index': test_idx,
        'y_true'      : y_test_np,
        'y_prob'      : p_test_np,
        'y_pred'      : y_pred_np,
        'threshold'   : best_train_thr,
    }).to_csv(f'FGC_GNN_{DATASET}_test_predictions_run{run_num + 1}.csv', index=False)


# Aggregate results across all runs 
results_df = pd.DataFrame(all_run_results)
results_df.to_csv(f'FGC_GNN_{DATASET}_all_runs_results.csv', index=False)

mean_auc  = results_df['test_auc'].mean()
std_auc   = results_df['test_auc'].std()
mean_f1   = results_df['test_f1'].mean()
std_f1    = results_df['test_f1'].std()
mean_acc  = results_df['test_acc'].mean()
std_acc   = results_df['test_acc'].std()
mean_prec = results_df['test_precision'].mean()
std_prec  = results_df['test_precision'].std()
mean_thr  = results_df['threshold'].mean()
std_thr   = results_df['threshold'].std()

print('\n' + '=' * 70)
print(f'  FINAL AGGREGATED RESULTS OVER {NUM_RUNS} RUNS  –  {DATASET}')
print('=' * 70)
print(f'  AUC        : {mean_auc:.4f} ± {std_auc:.4f}')
print(f'  F1 Score   : {mean_f1:.4f} ± {std_f1:.4f}')
print(f'  Accuracy   : {mean_acc:.4f} ± {std_acc:.4f}')
print(f'  Precision  : {mean_prec:.4f} ± {std_prec:.4f}')
print(f'  Threshold  : {mean_thr:.3f} ± {std_thr:.3f}')
print('=' * 70)

print(f'\nSaved files:')
print(f'  FGC_GNN_{DATASET}_cv_results.csv')
print(f'  FGC_GNN_{DATASET}_all_runs_results.csv')
for run_num in range(NUM_RUNS):
    print(f'  FGC_GNN_{DATASET}_final_training_curve_run{run_num + 1}.csv')
    print(f'  FGC_GNN_{DATASET}_test_predictions_run{run_num + 1}.csv')
    print(f'  FGC_GNN_{DATASET}_best_final_run{run_num + 1}.pt')

Imports OK
PyTorch   : 2.6.0+cu124
CUDA avail: True
Preprocessing complete.
  Patients : 139
  Features : 5640
  Omics splits: [1410, 1410, 1410, 1410]
  Label distribution:
1.0    86
0.0    53
Name: count, dtype: int64
Graph built.
  Nodes : 139
  Edges : 2780
FGC_GNN defined.
Param count (default config): 3,863,557
FocalLoss defined.
Device: cuda


Sanity-check training: 100%|█████████████████████████████████████████████████████████| 400/400 [01:11<00:00,  5.58it/s]



Best loss : 0.0029
Final AUC : 1.0000
Total combinations: 288
CV helpers defined.
Dataset  : LAML
Device   : cuda
Nodes    : 139
Edges    : 2780
in_dims  : [1410, 1410, 1410, 1410]
Combos   : 288

Using Repeated Stratified CV: 5-fold × 3 repeats = 15 total folds

[   1/288] hidden= 32 lr=1e-03 wd=1e-03 drop=0.3 epochs= 100 F1=0.7356±0.1218 ACC=0.6160±0.0590 THR=0.379±0.067
[   2/288] hidden= 32 lr=1e-03 wd=1e-03 drop=0.3 epochs= 200 F1=0.7467±0.0667 ACC=0.6069±0.0498 THR=0.385±0.087
[   3/288] hidden= 32 lr=1e-03 wd=1e-03 drop=0.3 epochs= 400 F1=0.7045±0.1979 ACC=0.6164±0.0794 THR=0.383±0.096
[   4/288] hidden= 32 lr=1e-03 wd=1e-03 drop=0.3 epochs= 100 F1=0.7663±0.0220 ACC=0.6238±0.0363 THR=0.356±0.054
[   5/288] hidden= 32 lr=1e-03 wd=1e-03 drop=0.3 epochs= 200 F1=0.7596±0.0188 ACC=0.6259±0.0247 THR=0.372±0.085
[   6/288] hidden= 32 lr=1e-03 wd=1e-03 drop=0.3 epochs= 400 F1=0.7129±0.1909 ACC=0.6069±0.0592 THR=0.357±0.090
[   7/288] hidden= 32 lr=1e-03 wd=1e-03 drop=0.3 epochs= 100 F1

Final Training (Run 1):   8%|████▋                                                    | 33/400 [00:05<01:06,  5.52it/s]



Early stopping triggered at epoch 34

Best training F1  : 1.0000
Best training AUC : 1.0000
Best training ACC : 1.0000
Best threshold    : 0.500

Run 1 Test Results (threshold=0.500):
  AUC       : 0.6471
  F1 Score  : 0.6000
  Accuracy  : 0.5714
  Precision : 0.6923

  RUN 2/10
Train samples : 111
Test  samples : 28
Train label dist: [42 69]
Test  label dist: [11 17]
Training final model | 400 epochs | patience=10


...


Early stopping triggered at epoch 28

Best training F1  : 1.0000
Best training AUC : 1.0000
Best training ACC : 1.0000
Best threshold    : 0.510

Run 8 Test Results (threshold=0.510):
  AUC       : 0.6524
  F1 Score  : 0.6667
  Accuracy  : 0.6429
  Precision : 0.7692

  RUN 9/10
Train samples : 111
Test  samples : 28
Train label dist: [42 69]
Test  label dist: [11 17]
Training final model | 400 epochs | patience=10


Final Training (Run 9):   8%|████▎                                                    | 30/400 [00:05<01:03,  5.80it/s]



Early stopping triggered at epoch 31

Best training F1  : 1.0000
Best training AUC : 1.0000
Best training ACC : 1.0000
Best threshold    : 0.580

Run 9 Test Results (threshold=0.580):
  AUC       : 0.6310
  F1 Score  : 0.5926
  Accuracy  : 0.6071
  Precision : 0.8000

  RUN 10/10
Train samples : 111
Test  samples : 28
Train label dist: [42 69]
Test  label dist: [11 17]
Training final model | 400 epochs | patience=10


Final Training (Run 10):   8%|████▊                                                   | 34/400 [00:05<01:04,  5.72it/s]



Early stopping triggered at epoch 35

Best training F1  : 1.0000
Best training AUC : 1.0000
Best training ACC : 1.0000
Best threshold    : 0.340

Run 10 Test Results (threshold=0.340):
  AUC       : 0.7380
  F1 Score  : 0.7568
  Accuracy  : 0.6786
  Precision : 0.7000

  FINAL AGGREGATED RESULTS OVER 10 RUNS  –  LAML
  AUC        : 0.5984 ± 0.0800
  F1 Score   : 0.6132 ± 0.1178
  Accuracy   : 0.5679 ± 0.0929
  Precision  : 0.6633 ± 0.0898
  Threshold  : 0.484 ± 0.082

Saved files:
  FGC_GNN_LAML_cv_results.csv
  FGC_GNN_LAML_all_runs_results.csv
  FGC_GNN_LAML_final_training_curve_run1.csv
  FGC_GNN_LAML_test_predictions_run1.csv
  FGC_GNN_LAML_best_final_run1.pt
  FGC_GNN_LAML_final_training_curve_run2.csv
  FGC_GNN_LAML_test_predictions_run2.csv
  FGC_GNN_LAML_best_final_run2.pt
  FGC_GNN_LAML_final_training_curve_run3.csv
  FGC_GNN_LAML_test_predictions_run3.csv
  FGC_GNN_LAML_best_final_run3.pt
  FGC_GNN_LAML_final_training_curve_run4.csv
  FGC_GNN_LAML_test_predictions_run4.csv
 